# Build a RAG Pipeline from Scratch

Before reaching for a framework, it helps to build a Retrieval-Augmented
Generation (RAG) pipeline by hand. This notebook uses only small, well-understood
building blocks (a PDF reader, an embedding model, NumPy, and the Groq client) so
that every step of the pipeline is visible and nothing is hidden behind an
abstraction. You will answer a question about the ReAct research paper using only
the content retrieved from that paper.

## Learning Objectives

By the end of this notebook, you should be able to:

- Describe the six stages of a RAG pipeline: load, chunk, embed, retrieve, augment, generate.
- Embed text locally and rank chunks by cosine similarity using NumPy.
- Assemble a context-grounded prompt and generate an answer with the Groq client.
- Recognise which parts a framework would later automate for you.
- Evaluate the result without labelled data, using retrieval scores and an LLM-as-judge RAG triad.

## The pipeline at a glance

<img src="assets/RAG_steps.png" width="700"/>

## Setup

Load the credentials and point at the source PDF. The Groq client reads
`GROQ_API_KEY` from the environment, which `load_dotenv` loads from your `.env` file.

In [1]:
import numpy as np
from dotenv import load_dotenv
from groq import Groq
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

load_dotenv()

PDF = "documents/react_paper.pdf"
QUERY = "What is the ReAct framework and how does it combine reasoning and acting?"

/Users/fritzvonscherzer/Documents/AIPM/aipm-rag-pipeline/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load the document

<img src="assets/RAG_step1_step2.png" width="800"/>

RAG starts with your own data. Here you read every page of the PDF and join the
text into a single string. `pypdf` extracts the raw text; in a production system
you might also keep page numbers so you can cite the source later.

In [2]:
reader = PdfReader(PDF)
full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
print(f"loaded {len(reader.pages)} pages, {len(full_text)} characters")

loaded 33 pages, 110106 characters


## Step 2: Split the text into chunks

Language models have a finite context window, and retrieval is more precise over
small passages than over a whole document. You therefore split the text into
overlapping windows. The overlap keeps sentences that straddle a boundary from
being cut in half.

This is the simplest possible splitter (fixed character windows). A framework would
offer smarter splitters that respect sentence or paragraph boundaries, but the idea
is the same.

In [3]:
def chunk_text(text, size=800, overlap=100):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return [c.strip() for c in chunks if c.strip()]


chunks = chunk_text(full_text)
print(f"created {len(chunks)} chunks")

created 158 chunks


## Step 3: Embed the chunks

<img src="assets/RAG_step3.png" width="400"/>

An embedding model turns each chunk into a vector that captures its meaning, so that
passages about similar topics sit close together in vector space. The model runs
locally through `sentence-transformers`, so no API key is needed for this step.

You normalise the vectors to unit length. With normalised vectors, the dot product
equals the cosine similarity, which makes the next step a single matrix multiplication.

In [4]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
chunk_vectors = model.encode(chunks, normalize_embeddings=True)
print(
    f"embedded {chunk_vectors.shape[0]} chunks into {chunk_vectors.shape[1]} dimensions"
)

embedded 158 chunks into 768 dimensions


## Step 4: Retrieve the most relevant chunks

<img src="assets/RAG_step4_step5.png" width="600"/>

To answer a question, embed the question with the same model, then score every chunk
by cosine similarity and keep the top matches. Because the vectors are normalised, the
cosine score is just the dot product, computed here with a single NumPy operation.

For a few hundred chunks, plain NumPy is fast enough. At larger scale you would store
the vectors in a dedicated vector store such as Chroma, which is exactly what the framework
notebooks do.

In [5]:
def retrieve(query, k=3):
    query_vector = model.encode([query], normalize_embeddings=True)[0]
    scores = chunk_vectors @ query_vector
    top = np.argsort(scores)[::-1][:k]
    return [chunks[i] for i in top]


context = retrieve(QUERY, k=3)
print(f"retrieved {len(context)} chunks for the query")

retrieved 3 chunks for the query


## Step 5: Augment the prompt

This is the heart of RAG. Instead of asking the model to answer from memory, you
build a prompt that contains the retrieved context and instruct the model to answer
from it. This grounds the answer in your document and reduces hallucination.

In [6]:
prompt = (
    "Answer the question using only the context below.\n\n"
    "Context:\n" + "\n---\n".join(context) + f"\n\nQuestion: {QUERY}\nAnswer:"
)
print(prompt[:600], end="...\n")

Answer the question using only the context below.

Context:
with ReAct, to allow a human to inspect and edit
ReAct’s reasoning traces. Figure 5 shows that by simply removing a hallucinating sentence in Act
17 and adding some hints in Act 23, ReAct can be made to change its behavior drastically to align
with these human thought edits and succeed in the task. From a human perspective, solving such a
task becomes signiﬁcantly easier, from typing tens of actions to only editing a couple of thoughts,
which enables new forms of human-machine collaboration. We note that such a policy edit on-the-go
1...


## Step 6: Generate the answer

Finally, send the augmented prompt to a Groq-hosted gpt-oss model. The model never sees
the full paper, only the handful of chunks you retrieved, yet it can answer the
question accurately because the relevant passages are in the prompt.

In [7]:
client = Groq()
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
)
print(response.choices[0].message.content)

**ReAct** is a framework that lets a language model alternate between *reasoning* (generating thoughts) and *acting* (executing actions).  
It combines the two by feeding the model’s own actions and the resulting observations back into the same input stream, so each new reasoning step can use the latest action outcome. This creates a coherent, sequential decision‑making process that is both interpretable and controllable—humans can inspect the reasoning trace and edit it on the fly to steer the agent’s behavior.


## Reading the result

The answer describes ReAct as a paradigm that interleaves reasoning traces with
actions, which is precisely what the paper introduces. The model produced this from
the retrieved chunks alone, which demonstrates the core promise of RAG: a general
model can answer questions about a specific document it was never trained on, as long
as you retrieve and supply the right context.

## Evaluating the answer

How do you know the pipeline worked? Without a labelled dataset you cannot compute ranking metrics such as precision or recall, but two things are still measurable from scratch: how confident the retrieval was, and how good the answer is when judged against its own context. The second uses the model itself as a judge, which is exactly what frameworks such as RAGAS automate.

### Retrieval confidence

The cosine scores from the retrieval step tell you how closely the top chunks matched the query. They are already computed inside `retrieve`, so here we recompute them to look at the numbers. A higher score means a closer semantic match.

In [8]:
query_vector = model.encode([QUERY], normalize_embeddings=True)[0]
scores = (
    chunk_vectors @ query_vector
)  # compute cosine similarity scores @ means matrix multiplication
top_scores = np.sort(scores)[::-1][:3]
print("Top-3 retrieval cosine scores:", [round(float(s), 3) for s in top_scores])
print("Mean of top 3:", round(float(top_scores.mean()), 3))

Top-3 retrieval cosine scores: [0.606, 0.586, 0.564]
Mean of top 3: 0.585


### The RAG triad (LLM as judge)

A reference-free way to judge quality is to ask the model to rate three things, one for each arrow of the pipeline:

1. **Context relevance**: is the retrieved context relevant to the question?
2. **Faithfulness**: is the answer grounded in that context, with no invented facts?
3. **Answer relevance**: does the answer actually address the question?

Each is a single call asking for a score from 1 to 5. This is the core idea behind tools like RAGAS, implemented here in a few lines.

In [9]:
import re


def llm_judge(instruction):
    """Ask the model to rate something from 1 to 5 and return the number."""
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": instruction + "\n\nReply with a single integer from 1 to 5.",
            }
        ],
        temperature=0,
    )
    content = response.choices[0].message.content
    if content is None:
        return None
    match = re.search(r"[1-5]", content)
    return int(match.group()) if match else None


answer = response.choices[0].message.content
joined_context = "\n---\n".join(context)

context_relevance = llm_judge(
    f"How relevant is this context to the question?\n\n"
    f"Question: {QUERY}\n\nContext:\n{joined_context}"
)
faithfulness = llm_judge(
    f"How well is the answer supported by the context, with no invented facts?\n\n"
    f"Context:\n{joined_context}\n\nAnswer: {answer}"
)
answer_relevance = llm_judge(
    f"How well does the answer address the question?\n\n"
    f"Question: {QUERY}\n\nAnswer: {answer}"
)

print("Context relevance:", context_relevance)
print("Faithfulness:     ", faithfulness)
print("Answer relevance: ", answer_relevance)

Context relevance: 5
Faithfulness:      5
Answer relevance:  5


These scores are a guide, not ground truth: an LLM judge can be biased, so treat them as one signal and read the answer yourself too. Metrics that need labelled data, such as precision, recall, or answer correctness against a reference answer, are where dedicated tools and evaluation datasets come in.

## Summary

You built a complete RAG pipeline from first principles:

- **Load** the PDF and **chunk** it into overlapping passages.
- **Embed** the chunks locally and **retrieve** the closest ones by cosine similarity.
- **Augment** a prompt with the retrieved context and **generate** a grounded answer.

Every stage was a few lines of plain Python. In the exercise that follows you will
apply this pipeline to a freshly converted and cleaned document, and the LangChain
notebook later rebuilds the same pipeline with a framework.

## References

- [Yao et al. (2022), ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)
- [sentence-transformers documentation](https://www.sbert.net/)
- [Groq API documentation](https://console.groq.com/docs)
- [A beginner's guide to building a RAG application from scratch](https://learnbybuilding.ai/tutorial/rag-from-scratch/)